In [1]:
!pip -q install pypdf2 pandas beautifulsoup4 lxml playwright
!playwright install chromium -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 7.7 MB/s eta 0:00:00
error: unknown option '-q'


In [5]:
import os, re, json, shutil
import pandas as pd
from datetime import datetime
from PyPDF2 import PdfReader
from bs4 import BeautifulSoup
from google.colab import drive
from playwright.async_api import async_playwright

drive.mount("/content/drive")

# ✅ Project root (you can change)
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"

# ✅ Inputs
PDF_DIR = os.path.join("/content/drive/MyDrive/Colab Notebooks/Shruti", "Data")   # <-- your PDFs folder in Drive

# ✅ Outputs (one folder per run)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", f"run_{RUN_ID}")

TEXT_OUT = os.path.join(OUT_DIR, "text_extracted")
REPORT_OUT = os.path.join(OUT_DIR, "reports")

os.makedirs(TEXT_OUT, exist_ok=True)
os.makedirs(REPORT_OUT, exist_ok=True)

print("✅ Project Ready")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PDF_DIR:", PDF_DIR)
print("RUN_ID:", RUN_ID)
print("TEXT_OUT:", TEXT_OUT)
print("REPORT_OUT:", REPORT_OUT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Project Ready
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project
PDF_DIR: /content/drive/MyDrive/Colab Notebooks/Shruti/Data
RUN_ID: 20260209_185402
TEXT_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/text_extracted
REPORT_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/reports


In [6]:
def clean_text_basic(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+\n", "\n\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def safe_filename(name: str) -> str:
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name).strip("_")
    return name[:120] if len(name) > 120 else name

def show_text_preview(title: str, text: str, n_chars: int = 800):
    print("\n" + "="*90)
    print("PREVIEW:", title)
    print("="*90)
    print(text[:n_chars])
    print("\n[Preview chars shown:", min(len(text), n_chars), "of", len(text), "]")


In [7]:
def get_drive_pdfs(pdf_dir: str):
    if not os.path.exists(pdf_dir):
        raise FileNotFoundError(f"PDF_DIR not found: {pdf_dir}")

    pdf_paths = [
        os.path.join(pdf_dir, f)
        for f in os.listdir(pdf_dir)
        if f.lower().endswith(".pdf")
    ]
    pdf_paths.sort()
    return pdf_paths

pdf_paths = get_drive_pdfs(PDF_DIR)

print("✅ PDFs Found in Drive:", len(pdf_paths))
for p in pdf_paths:
    print(" -", os.path.basename(p))


✅ PDFs Found in Drive: 2
 - Farming Schemes.pdf
 - farmerbook.pdf


In [8]:
def extract_pdf_text(pdf_file: str, preview_chars: int = 500, clean: bool = True):
    reader = PdfReader(pdf_file)
    num_pages = len(reader.pages)

    combined_text = []
    report_rows = []

    for i in range(num_pages):
        page_text = reader.pages[i].extract_text() or ""

        if clean:
            page_text = clean_text_basic(page_text)

        combined_text.append(page_text)

        char_count = len(page_text)
        word_count = len(page_text.split()) if page_text else 0
        preview = page_text[:preview_chars].replace("\n", " ")
        if char_count > preview_chars:
            preview += "..."

        report_rows.append({
            "page": i + 1,
            "chars_extracted": char_count,
            "words_extracted": word_count,
            "preview": preview
        })

    final_text = "\n\n".join([t for t in combined_text if t.strip()])
    report_df = pd.DataFrame(report_rows)
    return final_text, report_df


def run_pdf_pipeline_from_drive(pdf_paths):
    print("\n" + "="*90)
    print("STEP: PDF EXTRACTION PIPELINE")
    print("="*90)

    if not pdf_paths:
        print("❌ No PDFs found. Put PDFs inside:", PDF_DIR)
        return pd.DataFrame()

    pdf_summary_rows = []

    for pdf_path in pdf_paths:
        base = os.path.splitext(os.path.basename(pdf_path))[0]
        print("\n" + "-"*90)
        print("Processing PDF:", base)

        full_text, report_df = extract_pdf_text(pdf_path)

        # Save extracted text
        out_txt = os.path.join(TEXT_OUT, f"{safe_filename(base)}_pdf.txt")
        with open(out_txt, "w", encoding="utf-8") as f:
            f.write(full_text)

        # Save report CSV
        report_csv = os.path.join(REPORT_OUT, f"pdf_page_report_{safe_filename(base)}.csv")
        report_df.to_csv(report_csv, index=False)

        # Presentable outputs
        top_pages = report_df.sort_values("chars_extracted", ascending=False).head(5)[["page","chars_extracted","words_extracted","preview"]]
        print("✅ Saved text:", out_txt)
        print("✅ Saved page-report:", report_csv)
        print("Total chars:", len(full_text), "| Total words:", len(full_text.split()), "| Pages:", len(report_df))

        print("\nTop 5 pages (by extracted chars):")
        display(top_pages)

        show_text_preview(f"{base} (PDF extracted text)", full_text, n_chars=700)

        pdf_summary_rows.append({
            "source_type": "pdf",
            "source_name": os.path.basename(pdf_path),
            "saved_text_file": out_txt,
            "pages": len(report_df),
            "total_chars": len(full_text),
            "total_words": len(full_text.split())
        })

    pdf_summary_df = pd.DataFrame(pdf_summary_rows)
    pdf_summary_path = os.path.join(REPORT_OUT, "pdf_summary.csv")
    pdf_summary_df.to_csv(pdf_summary_path, index=False)

    print("\n✅ PDF PIPELINE COMPLETE")
    print("Summary saved:", pdf_summary_path)
    display(pdf_summary_df)

    return pdf_summary_df

pdf_summary_df = run_pdf_pipeline_from_drive(pdf_paths)



STEP: PDF EXTRACTION PIPELINE

------------------------------------------------------------------------------------------
Processing PDF: Farming Schemes
✅ Saved text: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/text_extracted/Farming_Schemes_pdf.txt
✅ Saved page-report: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/reports/pdf_page_report_Farming_Schemes.csv
Total chars: 638828 | Total words: 94954 | Pages: 314

Top 5 pages (by extracted chars):


,page,chars_extracted,words_extracted,preview
265,266,3371,500,265 Publicity and Brand promotion shall help h...
156,157,3299,569,156 The key activities in HBNC constitute the ...
234,235,3247,506,234 limited to a village in Rural Areas or a w...
173,174,3241,472,173 The Scheme will be implemented as a Centra...
153,154,3197,490,153 Facility based newborn screening This incl...



PREVIEW: Farming Schemes (PDF extracted text)
COMPENDIUM 
OF 
GOVERNMENT OF INDIA SCHEMES AND 
PROGRAMMES RELEVANT TO THE 
NORTH EASTERN STATES 

 

 

 

 

Compiled by: 
NERCORMP 
NORTH EASTERN REGION COMMUNITY RESOURCE 
MANAGEMENT PROJECT 
SHILLONG - 793 003 

November 2020

1 CONTENTS 

 
Name of Schemes Page No. 

1. Ministry of Agriculture & Farmers Welfare 
1.1 Minimum Support Price Scheme 10 
1.2 Marketing research and information network (under Integrated Scheme for 
Agricultural Marketing -ISAM) 11 
1.3 Central Herd Registration Scheme 12 
1.4 National Food Security Mission 13 
1.5 Rashtriya Krishi Vikas Yojana Remunerative Approaches for Agriculture 
and Allied sector Rejuvenation (RKVY -RAFTAAR) 15 
1.6 National Mission for 

[Preview chars shown: 700 of 638828 ]

------------------------------------------------------------------------------------------
Processing PDF: farmerbook
✅ Saved text: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_202

,page,chars_extracted,words_extracted,preview
5,6,4938,720,Farmer’s Handbook on Basic Agriculture Preface...
135,136,3942,655,128 Occupational Health and Safety Farmer’s Ha...
85,86,3788,583,78 Plant Protection Farmer’s Handbook on Basic...
73,74,3654,594,66 Soil and Plant Nutrition Farmer’s Handbook ...
17,18,3648,605,10 General Conditions for Cultivation of Crops...



PREVIEW: farmerbook (PDF extracted text)
A holistic 
perspective of 
scientific 
agriculture
A joint initiative to 
impart farmers with 
technical knowledge on 
basic agriculture.Farmer’s Handbook on Basic Agriculture

Disclaimer:
The opinions expressed provided in this publication are those of the authors and do not necessarily reflect those of GIZ . The 
designations employed and the presentation of material in this publication do not imply the expression of any opinion what-soever on the part of GIZ concerning the legal status of any country, territory, city or area, or concerning the delimitation of its frontiers or boundaries.

Farmer’s Handbook on Basic Agriculture
Farmer’s Handbook on Basic Agriculture
Prepared & compiled by

[Preview chars shown: 700 of 278092 ]

✅ PDF PIPELINE COMPLETE
Summary saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/reports/pdf_summary.csv


,source_type,source_name,saved_text_file,pages,total_chars,total_words
0,pdf,Farming Schemes.pdf,/content/drive/MyDrive/Colab Notebooks/Shruti/...,314,638828,94954
1,pdf,farmerbook.pdf,/content/drive/MyDrive/Colab Notebooks/Shruti/...,154,278092,43759


In [16]:
!pip -q install playwright beautifulsoup4 lxml
!playwright install --with-deps chromium


Installing dependencies...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,396 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,720 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-sec

In [17]:
import os, subprocess, sys

def ensure_playwright_chromium():
    cache_dir = os.path.expanduser("~/.cache/ms-playwright")
    has_chromium = os.path.exists(cache_dir) and any("chromium" in x for x in os.listdir(cache_dir))

    if not has_chromium:
        print("⚙️ Playwright Chromium not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "playwright"])
        subprocess.check_call(["playwright", "install", "chromium"])  # no -q here
        print("✅ Chromium installed.")
    else:
        print("✅ Playwright Chromium already installed.")

ensure_playwright_chromium()


✅ Playwright Chromium already installed.


In [18]:
def extract_text_headings_links(html: str):
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    headings = []
    for h in soup.find_all(["h1", "h2", "h3"]):
        t = h.get_text(" ", strip=True)
        if t:
            headings.append(t)

    links = []
    for a in soup.find_all("a", href=True):
        text = a.get_text(" ", strip=True)
        href = a["href"].strip()
        if text and href:
            links.append((text, href))

    visible_text = soup.get_text("\n", strip=True)
    visible_text = clean_text_basic(visible_text)
    return visible_text, headings, links


async def fetch_rendered_page(page, url: str, wait_ms: int = 5000):
    resp = await page.goto(url, wait_until="domcontentloaded", timeout=60000)
    await page.wait_for_timeout(wait_ms)
    try:
        await page.wait_for_load_state("networkidle", timeout=15000)
    except:
        pass
    html = await page.content()
    return html, (resp.status if resp else None), page.url


async def discover_section_urls(main_url: str, section_keywords: list, base: str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        html, status, final_url = await fetch_rendered_page(page, main_url, wait_ms=6000)
        text, headings, links = extract_text_headings_links(html)

        section_urls = {}
        for label, href in links:
            if label in section_keywords:
                if href.startswith("/"):
                    full = base + href
                elif href.startswith("http"):
                    full = href
                else:
                    full = base + "/" + href
                section_urls[label] = full

        await browser.close()

    return section_urls, {
        "status": status,
        "final_url": final_url,
        "headings": headings[:20],
        "text_preview": text[:400]
    }


async def extract_all_sections(section_urls: dict):
    results = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for title, url in section_urls.items():
            html, status, final_url = await fetch_rendered_page(page, url, wait_ms=6000)
            text, headings, _ = extract_text_headings_links(html)

            results.append({
                "section": title,
                "url": url,
                "final_url": final_url,
                "status": status,
                "rendered_html_chars": len(html),
                "text_chars": len(text),
                "text_words": len(text.split()),
                "headings_count": len(headings),
                "text": text,
                "headings": headings[:30]
            })

        await browser.close()

    report_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ["text","headings"]} for r in results])
    return results, report_df


async def run_web_pipeline_presentable():
    print("\n" + "="*90)
    print("STEP: WEB EXTRACTION PIPELINE")
    print("="*90)

    BASE = "https://nsc.mospi.gov.in"
    MAIN_URL = "https://nsc.mospi.gov.in/4-agricultural-statistics"

    section_keywords = [
        "Introduction",
        "Crop Area Statistics",
        "Crop Production",
        "Crop Forecasts",
        "Production of Horticultural Crops",
        "Land Use",
        "Irrigation Statistics",
        "Agricultural Prices",
        "Fisheries Statistics",
        "Forestry Statistics"
    ]

    section_urls, main_info = await discover_section_urls(MAIN_URL, section_keywords, BASE)

    print("Main page:", main_info["final_url"], "| status:", main_info["status"])
    print("Sections found:", len(section_urls))
    for k,v in section_urls.items():
        print("-", k, "->", v)

    section_results, report_df = await extract_all_sections(section_urls)

    # Save combined + per-section text
    combined_web_text = "\n\n".join(
        [f"SECTION: {r['section']}\nURL: {r['final_url']}\n\n{r['text']}" for r in section_results]
    )

    for r in section_results:
        fname = f"web_{safe_filename(r['section'])}.txt"
        path = os.path.join(TEXT_OUT, fname)
        with open(path, "w", encoding="utf-8") as f:
            f.write(f"SECTION: {r['section']}\nURL: {r['final_url']}\n\n{r['text']}")

    combined_path = os.path.join(TEXT_OUT, "web_ALL.txt")
    with open(combined_path, "w", encoding="utf-8") as f:
        f.write(combined_web_text)

    report_path = os.path.join(REPORT_OUT, "web_sections_report.csv")
    report_df.to_csv(report_path, index=False)

    meta_path = os.path.join(REPORT_OUT, "web_main_info.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(main_info, f, indent=2)

    print("\n✅ WEB PIPELINE COMPLETE")
    print("Saved combined web text:", combined_path)
    print("Saved web report:", report_path)
    print("Saved web metadata:", meta_path)

    # Presentable outputs: top sections by text length
    top_sections = report_df.sort_values("text_chars", ascending=False).head(5)
    print("\nTop sections (by extracted chars):")
    display(top_sections)

    # Preview first section text
    if section_results:
        show_text_preview(f"WEB: {section_results[0]['section']}", section_results[0]["text"], n_chars=700)

    return report_df, combined_path

web_report_df, web_all_path = await run_web_pipeline_presentable()
display(web_report_df)



STEP: WEB EXTRACTION PIPELINE
Main page: https://nsc.mospi.gov.in/4-agricultural-statistics | status: 200
Sections found: 10
- Introduction -> https://nsc.mospi.gov.in/41-introduction
- Crop Area Statistics -> https://nsc.mospi.gov.in/42-crop-area-statistics
- Crop Production -> https://nsc.mospi.gov.in/43-crop-production
- Crop Forecasts -> https://nsc.mospi.gov.in/44-crop-forecasts
- Production of Horticultural Crops -> https://nsc.mospi.gov.in/45-production-horticultural-crops
- Land Use -> https://nsc.mospi.gov.in/47-land-use
- Irrigation Statistics -> https://nsc.mospi.gov.in/48-irrigation-statistics
- Agricultural Prices -> https://nsc.mospi.gov.in/410-agricultural-prices
- Fisheries Statistics -> https://nsc.mospi.gov.in/416-fisheries-statistics
- Forestry Statistics -> https://nsc.mospi.gov.in/417-forestry-statistics

✅ WEB PIPELINE COMPLETE
Saved combined web text: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/text_extracted/web_

,section,url,final_url,status,rendered_html_chars,text_chars,text_words,headings_count
1,Crop Area Statistics,https://nsc.mospi.gov.in/42-crop-area-statistics,https://nsc.mospi.gov.in/42-crop-area-statistics,200,1649592,17252,2778,4
2,Crop Production,https://nsc.mospi.gov.in/43-crop-production,https://nsc.mospi.gov.in/43-crop-production,200,1642334,10187,1626,4
6,Irrigation Statistics,https://nsc.mospi.gov.in/48-irrigation-statistics,https://nsc.mospi.gov.in/48-irrigation-statistics,200,1640788,8619,1318,4
0,Introduction,https://nsc.mospi.gov.in/41-introduction,https://nsc.mospi.gov.in/41-introduction,200,1639301,7296,1066,4
3,Crop Forecasts,https://nsc.mospi.gov.in/44-crop-forecasts,https://nsc.mospi.gov.in/44-crop-forecasts,200,1638659,6549,981,4



PREVIEW: WEB: Introduction
NSC
SKIP TO MAIN CONTENT
|
SCREEN READER ACCESS
|
Accessibility Controls
Home
About us
Documents
RTI
Gallery
Contact Us
Home
About us
Documents
RTI
Gallery
Contact Us
Home
/
Report of Dr. Rangarajan Commission
Documents
Reports
4.1 INTRODUCTION
Central Statistical Organisation
4.1.1 Agriculture plays a vital role in the Indian economy. Over 70 per cent of the rural households depend on agriculture as their principal means of livelihood. Agriculture along with fisheries and forestry accounts for one-third of the nation’s Gross Domestic Product (GDP) and is its single largest contributor. Agricultural exports constitute a fifth of the total exports of the country. In view of the predominant po

[Preview chars shown: 700 of 7296 ]


,section,url,final_url,status,rendered_html_chars,text_chars,text_words,headings_count
0,Introduction,https://nsc.mospi.gov.in/41-introduction,https://nsc.mospi.gov.in/41-introduction,200,1639301,7296,1066,4
1,Crop Area Statistics,https://nsc.mospi.gov.in/42-crop-area-statistics,https://nsc.mospi.gov.in/42-crop-area-statistics,200,1649592,17252,2778,4
2,Crop Production,https://nsc.mospi.gov.in/43-crop-production,https://nsc.mospi.gov.in/43-crop-production,200,1642334,10187,1626,4
3,Crop Forecasts,https://nsc.mospi.gov.in/44-crop-forecasts,https://nsc.mospi.gov.in/44-crop-forecasts,200,1638659,6549,981,4
4,Production of Horticultural Crops,https://nsc.mospi.gov.in/45-production-horticu...,https://nsc.mospi.gov.in/45-production-horticu...,200,1638146,6048,917,4
5,Land Use,https://nsc.mospi.gov.in/47-land-use,https://nsc.mospi.gov.in/47-land-use,200,1636941,4865,766,4
6,Irrigation Statistics,https://nsc.mospi.gov.in/48-irrigation-statistics,https://nsc.mospi.gov.in/48-irrigation-statistics,200,1640788,8619,1318,4
7,Agricultural Prices,https://nsc.mospi.gov.in/410-agricultural-prices,https://nsc.mospi.gov.in/410-agricultural-prices,200,1638266,6106,944,4
8,Fisheries Statistics,https://nsc.mospi.gov.in/416-fisheries-statistics,https://nsc.mospi.gov.in/416-fisheries-statistics,200,1638631,6502,1005,4
9,Forestry Statistics,https://nsc.mospi.gov.in/417-forestry-statistics,https://nsc.mospi.gov.in/417-forestry-statistics,200,1638199,6063,935,4


In [19]:
print("\n" + "="*90)
print("STEP: FINAL COMBINED SUMMARY")
print("="*90)

combined_summary = []

# PDFs
if len(pdf_summary_df) > 0:
    for _, row in pdf_summary_df.iterrows():
        combined_summary.append({
            "source_type": "pdf",
            "source_name": row["source_name"],
            "total_chars": int(row["total_chars"]),
            "total_words": int(row["total_words"]),
            "saved_text_file": row["saved_text_file"]
        })

# Web combined
if os.path.exists(web_all_path):
    with open(web_all_path, "r", encoding="utf-8") as f:
        web_text = f.read()
    combined_summary.append({
        "source_type": "web",
        "source_name": "NSC_MoSPI_4_Agricultural_Statistics_ALL",
        "total_chars": len(web_text),
        "total_words": len(web_text.split()),
        "saved_text_file": web_all_path
    })

final_df = pd.DataFrame(combined_summary)
final_report_path = os.path.join(REPORT_OUT, "dataset_prep_final_summary.csv")
final_df.to_csv(final_report_path, index=False)

print("✅ FINAL SUMMARY SAVED:", final_report_path)
display(final_df)

# ZIP everything from this run (for guide submission)
zip_path = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("\n✅ RUN OUTPUTS ZIPPED:", zip_path)



STEP: FINAL COMBINED SUMMARY
✅ FINAL SUMMARY SAVED: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/reports/dataset_prep_final_summary.csv


,source_type,source_name,total_chars,total_words,saved_text_file
0,pdf,Farming Schemes.pdf,638828,94954,/content/drive/MyDrive/Colab Notebooks/Shruti/...
1,pdf,farmerbook.pdf,278092,43759,/content/drive/MyDrive/Colab Notebooks/Shruti/...
2,web,NSC_MoSPI_4_Agricultural_Statistics_ALL,80317,12388,/content/drive/MyDrive/Colab Notebooks/Shruti/...



✅ RUN OUTPUTS ZIPPED: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402.zip
